In [1]:
# Configuration AWS
BEDROCK_MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"
BEDROCK_EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"
AWS_REGION = "us-east-1"

In [70]:
# OpenSearch Configuration
OPENSEARCH_URL = "https://pyx7grd5myluqvsirhji.ap-south-1.aoss.amazonaws.com"
OPENSEARCH_HOST ="pyx7grd5myluqvsirhji.ap-south-1.aoss.amazonaws.com"
INDEX_NAME = "rag-bedrock-index"

In [74]:
# Imports
import os
import boto3
from opensearchpy import RequestsHttpConnection, AWSV4SignerAuth

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import OpenSearchVectorSearch
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings
from langchain_aws import ChatBedrock

In [75]:
# OpenSearch Auth
credentials = boto3.Session().get_credentials()

awsauth = AWSV4SignerAuth(
    credentials,
    AWS_REGION,
    "es"
)

In [ ]:
# Testing AWS Client
test_llm = ChatBedrock(
    model_id = BEDROCK_MODEL_ID,
    region_name = AWS_REGION,
    model_kwargs = {
        "max_tokens": 1000,
        "temperature": 0.7,
        "top_p": 0.9
    }
)
test_response = test_llm.invoke("Explain what does ML acutally mean in 1 sentence.")
print("Response:", test_response.content)

Response: Machine Learning (ML) is a field of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed.


In [68]:
# Initialize Embeddings
embedding = BedrockEmbeddings(
    model_id=BEDROCK_EMBEDDING_MODEL_ID,
    region_name=AWS_REGION
)

In [45]:
# Load Metadata of Books
import json

# Open the file in read mode
with open('../Books/metadata-book.json', 'r') as file:
    data = json.load(file)

BOOK_INFO = data
json_formatted_str = json.dumps(data, indent=4)

print(json_formatted_str)



{
    "ml_oreilly.pdf": {
        "book_title": "Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow",
        "author": "Aurelien Geron",
        "publisher": "O'Reilly"
    },
    "ml_alpaydin.pdf": {
        "book_title": "Introduction to Machine Learning",
        "author": "Ethem Alpaydin",
        "publisher": "MIT Press"
    }
}


In [47]:
# Checking pdfs from folders
BOOK_FOLDER = "../Books"

for file in os.listdir(BOOK_FOLDER):

    if not file.endswith(".pdf"):
        continue
    else:
        print(file)

ml_alpaydin.pdf
ml_oreilly.pdf


In [51]:
#  Load PDFs + Chunk + Attach Metadata
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

documents = []

for file in os.listdir(BOOK_FOLDER):

    if not file.endswith(".pdf"):
        continue

    print("Processing:", file)

    loader = PyPDFLoader(os.path.join(BOOK_FOLDER, file))
    pages = loader.load()

    chunks = splitter.split_documents(pages)

    meta = BOOK_INFO.get(file, {})

    for i, chunk in enumerate(chunks):

        metadata = {
            "book_title": meta.get("book_title"),
            "author": meta.get("author"),
            "publisher": meta.get("publisher"),
            "source_file": file,
            "page": chunk.metadata.get("page"),
            "chunk_id": f"{file}_{i}"
        }

        documents.append(
            Document(
                page_content=chunk.page_content,
                metadata=metadata
            )
        )

print("Total chunks created:", len(documents))

Processing: ml_alpaydin.pdf
Processing: ml_oreilly.pdf
Total chunks created: 1808


## FAISS LOCAL VECTOR STORE

In [58]:
# Backup Locally ( FAISS)

vector_store = FAISS.from_documents(
    documents,
    embedding
)

print("Saving FAISS index locally...")

vector_store.save_local("faiss_index")
print("FAISS index saved successfully.")

Saving FAISS index locally...
FAISS index saved successfully.


In [ ]:
# Verifying FAISS
vector_store = FAISS.load_local(
    "faiss_index",
    embedding,
    allow_dangerous_deserialization=True
)

# A sentence from book Introduction to Machine Learning (ml_alphaydin.pdf) , page no 302
results = vector_store.similarity_search(
    "A perceptron that has a single layer of weights can only approximate linear functions of the input and cannot solve problems like the XOR, where the discrimininant to be estimated is nonlinear.",
    k=3
)

for r in results:
    print(r.page_content)
    print(r.metadata)

11.5 Multilayer Perceptrons 279
Table 11.2 Input and output for the XOR function
x1 x2 r
0 0 0
0 1 1
1 0 1
1 1 0
x1
x2
Figure 11.5 XOR problem is not linearly separable. We cannot draw a line where
the empty circles are on one side and the ﬁlled circles on the other side.
This result should not be very surprising to us since the VC dimension
of a line (in two dimensions) is three. With two binary inputs there are
four cases, and thus we know that there exist problems with two inputs
that are not solvable using a line; XOR is one of them.
11.5 Multilayer Perceptrons
A perceptron that has a single layer of weights can only approximate lin-
ear functions of the input and cannot solve problems like the XOR, where
the discrimininant to be estimated is nonlinear. Similarly, a perceptron
{'book_title': 'Introduction to Machine Learning', 'author': 'Ethem Alpaydin', 'publisher': 'MIT Press', 'source_file': 'ml_alpaydin.pdf', 'page': 302, 'chunk_id': 'ml_alpaydin.pdf_838'}
are solvable using th

In [63]:
#  Checking with LLM ( with faiss )

FAISS_PATH = "faiss_index"

vector_store = FAISS.load_local(
    FAISS_PATH,
    embedding,
    allow_dangerous_deserialization=True
)
test_llm = ChatBedrock(
    model_id = BEDROCK_MODEL_ID,
    region_name = AWS_REGION,
    model_kwargs = {
        "max_tokens": 1000,
        "temperature": 0.7,
        "top_p": 0.9
    }
)

retriever = vector_store.as_retriever( search_kwargs={"k": 3} )

query = "What is gradient descent?"
docs = retriever.invoke(query)

print("\nRetrieved Documents:\n")

for d in docs:
    print(d.metadata)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a helpful assistant tasked with answering questions based strictly on the provided context. Follow these guidelines precisely:

CONTEXT USAGE:
- Answer ONLY using information from the provided context below
- If the context doesn't contain relevant information, state "I cannot answer this question based on the provided context"
- Do not use any external knowledge or make assumptions beyond the context

FORMATTING REQUIREMENTS:
- For mathematical expressions and formulas, use LaTeX notation with $$ delimiters (e.g., $$E = mc^2$$)
- For inline formulas, you may use $ delimiters (e.g., $x^2 + y^2 = z^2$)
- Structure your response with clear paragraphs and bullet points where appropriate
- If the context includes code, preserve its original formatting

QUALITY GUIDELINES:
- Be concise but thorough in your response
- Quote or reference specific parts of the context when relevant
- If the context presents multiple perspectives, acknowledge them
- Maintain academic integrity by not adding information outside the context

CONTEXT:
{context}

QUESTION:
{query}

INSTRUCTIONS:
1. First, analyze if the context contains information to answer the question
2. If yes, provide a well-structured answer using only the context
3. If no, clearly state the limitation
4. Format all mathematical expressions in LaTeX
5. Keep the response focused and directly relevant to the question

ANSWER:
"""


# Generate Response

response = test_llm.invoke(prompt)

print("\nFinal Answer:\n")
print(response.content)


Retrieved Documents:

{'book_title': 'Introduction to Machine Learning', 'author': 'Ethem Alpaydin', 'publisher': 'MIT Press', 'source_file': 'ml_alpaydin.pdf', 'page': 272, 'chunk_id': 'ml_alpaydin.pdf_759'}
{'book_title': 'Introduction to Machine Learning', 'author': 'Ethem Alpaydin', 'publisher': 'MIT Press', 'source_file': 'ml_alpaydin.pdf', 'page': 272, 'chunk_id': 'ml_alpaydin.pdf_758'}
{'book_title': 'Introduction to Machine Learning', 'author': 'Ethem Alpaydin', 'publisher': 'MIT Press', 'source_file': 'ml_alpaydin.pdf', 'page': 298, 'chunk_id': 'ml_alpaydin.pdf_829'}

Final Answer:

Based on the provided context, gradient descent is an iterative optimization method used to minimize a differentiable function E(w) of a vector of variables w. The key aspects of gradient descent are:

1. The gradient vector ∇wE is composed of the partial derivatives of E with respect to each element of w:

$$\nabla_wE = \left[\frac{\partial E}{\partial w_1}, \frac{\partial E}{\partial w_2}, ..., 


Based on the provided context, gradient descent is an iterative optimization method used to minimize a differentiable function E(w) of a vector of variables w. The key aspects of gradient descent are:

1. The gradient vector ∇wE is composed of the partial derivatives of E with respect to each element of w:

$$\nabla_wE = \left[\frac{\partial E}{\partial w_1}, \frac{\partial E}{\partial w_2}, ..., \frac{\partial E}{\partial w_d}\right]^T$$

2. The gradient descent procedure starts from a random initial value of w, and at each step, updates w in the opposite direction of the gradient:

$$\Delta w_i = -\eta \frac{\partial E}{\partial w_i}, \forall i$$
$$w_i = w_i + \Delta w_i$$

where η is the step size or learning rate, which determines the magnitude of the update.

3. Gradient descent is used to minimize the error function E(w), by iteratively updating the parameters w in the direction that decreases E the fastest.

The context also mentions that in online learning, the error function is evaluated on individual instances, and the parameters are updated using stochastic gradient descent, where the updates are performed after each instance.

## AWS OpenSearch Service as Vector Store

In [ ]:
# Create OpenSearch Vector Store
vector_store = OpenSearchVectorSearch.from_documents(
    documents,
    embedding,
    opensearch_url=f"https://{OPENSEARCH_HOST}",
    http_auth=awsauth,
    index_name=INDEX_NAME,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    bulk_size=2000
)